# 选修E10 · Day 1 上机：Agent经济基础--Agent作为经济主体

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **mesa** 构建Agent经济仿真--买方Agent/卖方Agent两类经济主体通过A2A协商交易，涌现市场价格/财富分布/存活率
2. 用 **networkx** 分析Agent交易网络拓扑--密度/聚类系数/PageRank经济影响力
3. 用 **numpy-financial** 计算Agent-as-Worker的NPV/IRR，量化Agent作为经济主体的投资价值
4. 理解Agent经济三层模型和贝叶斯Agent决策--Agent用贝叶斯更新估计公平价格
5. 建立天道推演×多Agent仿真的同构认知--仿真本质是计算化的天道推演沙盘

## 真实库与真实数据
- **mesa**（agent-based modeling 框架）：构建Agent经济仿真
- **networkx**（图网络分析）：Agent交易网络拓扑
- **numpy-financial**（金融计算）：Agent经济价值NPV/IRR
- **pandas + matplotlib**：仿真结果分析与可视化
- **真实经济参数**：A2A协议费10%、Token定价$5/1M（GPT-4o真实定价）、推理成本约束

> 详见 data/README.md

## 0. 环境准备

首次运行需安装依赖（取消注释执行一次）：

> 所有库（mesa/networkx/numpy-financial/pandas/matplotlib/numpy）均为本地可用库，不需要API Key。

In [ ]:
# !pip install mesa networkx numpy-financial pandas matplotlib numpy -q

import warnings
warnings.filterwarnings('ignore')

import mesa
import numpy as np
import pandas as pd
import networkx as nx
import numpy_financial as npf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mesa.datacollection import DataCollector

print(f"mesa {mesa.__version__} | networkx {nx.__version__}")
print("Agent经济仿真环境就绪")

## 1. 真实经济参数

Agent经济仿真的参数基于真实世界的经济数据：

| 参数 | 值 | 真实来源 |
|------|-----|---------|
| A2A协议费率 | 10% | Agent间去中心化交易协议费率 |
| Token定价 | $5/1M tokens | GPT-4o input 定价（OpenAI 2024-2025定价页） |
| 每次A2A协商推理token | 500 tokens | Agent协商/比价/决策的合理token消耗 |
| 推理成本/协商 | ~$0.0025 | 500 tokens × $5/1M |

**推理成本是Agent经济的核心约束**--AI Agent每次A2A协商都消耗token。

In [ ]:
# 真实经济参数（可追溯来源）
# A2A协议费率 10%: 去中心化Agent交易协议费率
A2A_PROTOCOL_FEE_RATE = 0.10
# Token定价: GPT-4o input ~$5/1M tokens (OpenAI 2024-2025定价页)
TOKEN_PRICE_PER_1M = 5.0
# 每次A2A协商推理token消耗
TOKENS_PER_NEGOTIATION = 500
# 每次A2A协商的推理成本
REASONING_COST_PER_NEGOTIATION = (TOKENS_PER_NEGOTIATION / 1_000_000) * TOKEN_PRICE_PER_1M

print(f"A2A协议费率: {A2A_PROTOCOL_FEE_RATE*100:.0f}%")
print(f"推理成本/协商: ${REASONING_COST_PER_NEGOTIATION:.4f}")

## 2. TODO 1：买方Agent（贝叶斯价格估计）

**买方Agent** 是Agent经济中的需求方（营销映射：品牌Agent买广告位）。

核心属性：
- `wealth`：预算（初始1000）
- `price_mu` / `price_var`：贝叶斯价格信念（Normal先验）
- `n_obs`：观测次数

**行为逻辑**：
1. 70%概率选择最低价卖方（exploit），30%概率随机选择（explore）
2. 只接受低于"后验均值+1标准差"的价格（贝叶斯决策）
3. 用观测价格更新贝叶斯后验（共轭正态更新）
4. 预算耗尽则破产

**贝叶斯更新**：Agent作为经济主体在不确定市场中学习公平价格。

In [ ]:
# TODO 1：买方Agent（贝叶斯价格估计）
# 提示：继承mesa.Agent
#   属性：wealth=1000, price_mu=20, price_var=25, n_obs=0, observation_var=10
#   方法：update_price_belief(price) -> 共轭正态后验更新
#   行为：70%exploit/30%explore, 贝叶斯价格上界判断, A2A购买, 破产检查

class BuyerAgent(mesa.Agent):
    # ===== 你的代码 =====

    # raise NotImplementedError

## 3. TODO 2：卖方Agent（A2A协商 + 推理成本）

**卖方Agent** 是Agent经济中的供给方（营销映射：媒介Agent卖广告流量）。

核心属性：
- `wealth`：资金（初始500）
- `base_cost`：基础成本（产品差异化，随机0.8-1.3倍）
- `price`：当前售价（动态调整）
- `supply`：供应量
- `a2a_negotiations`：A2A协商次数
- `total_reasoning_cost`：累计推理成本

**行为逻辑**：
1. A2A交易：收取价格，支付10%协议费 + 推理成本
2. 供应高则降价，供应低则涨价（动态定价）
3. 定期补货（消耗资金）
4. 资金为负则破产

**真实参数**：A2A协议费10% + 推理成本$0.0025/协商。

In [ ]:
# TODO 2：卖方Agent（A2A协商 + 推理成本）
# 提示：继承mesa.Agent
#   属性：wealth=500, base_cost=10*(0.8-1.3随机), price=base_cost*(1.5-2.5随机)
#   方法：sell_to(buyer) -> 收价格, 支付10%协议费+推理成本, 动态调价
#   行为：补货 + 破产检查

class SellerAgent(mesa.Agent):
    # ===== 你的代码 =====

    # raise NotImplementedError

## 4. TODO 3：Agent交易网络（networkx）

**AgentTransactionNetwork** 用networkx分析Agent间A2A交易拓扑。

核心功能：
- `add_transaction(buyer_id, seller_id, amount)`：添加有向交易边
- `compute_metrics()`：计算网络密度/聚类系数/边数/节点数
- `top_sellers_by_pagerank(top_n)`：PageRank经济影响力排名

**经济意义**：网络密度反映市场交易紧密程度，PageRank识别经济hub Agent。

In [ ]:
# TODO 3：Agent交易网络（networkx）
# 提示：用nx.DiGraph()构建有向图
#   add_transaction: 添加/更新 buyer->seller 边 (weight=累计金额, count=交易次数)
#   compute_metrics: 返回 density/avg_clustering/n_edges/n_nodes
#   top_sellers_by_pagerank: 用nx.pagerank排序, 返回top_n [(node_id, score)]

class AgentTransactionNetwork:
    # ===== 你的代码 =====

    # raise NotImplementedError

## 5. TODO 4：Agent经济模型 + DataCollector

**AgentEconomyModel** 整合买方/卖方Agent和交易网络，用DataCollector追踪涌现指标：

| 指标 | 含义 |
|------|------|
| `gini` | 基尼系数（财富不平等程度） |
| `avg_price` | 市场平均价格 |
| `price_std` | 价格标准差 |
| `n_alive_*` | 各类Agent存活数 |
| `total_a2a_trades` | 累计A2A交易量 |
| `network_density` | 交易网络密度 |
| `total_reasoning_cost` | 累计推理成本 |

**天道推演映射**：模型每一步step()就是一次沙盘推演。

In [ ]:
# TODO 4：Agent经济模型 + DataCollector
# 提示：继承mesa.Model
#   __init__: 创建20买方+5卖方 + AgentTransactionNetwork + DataCollector(9个model_reporters+3个agent_reporters)
#   _compute_gini: G = 2*sum((i+1)*w)/(n*sum(w)) - (n+1)/n
#   step: agents.shuffle_do("step") + datacollector.collect

class AgentEconomyModel(mesa.Model):
    # ===== 你的代码 =====

    # raise NotImplementedError

## 6. TODO 5：运行仿真 + 提取数据

运行Agent经济仿真20个tick，用DataCollector提取时间序列数据到pandas DataFrame。

**关键问题**：
- 基尼系数如何变化？（财富是否越来越集中？）
- 市场价格是否收敛？
- 交易网络拓扑如何演化？
- 哪类Agent最先破产？

In [ ]:
# TODO 5：运行仿真 + 提取数据
# 提示：运行20步，用datacollector提取model_vars和agent_vars到DataFrame
#   打印仿真规模、最终基尼、价格分布、存活数、A2A交易量、网络拓扑

# ===== 你的代码 =====

# raise NotImplementedError

## 7. TODO 6：Agent经济价值分析（NPV/IRR）+ 可视化

用 **numpy-financial** 计算Agent-as-Worker的经济价值，用matplotlib绘制4个子图：

1. **NPV/IRR分析**：对比Agent-as-Worker vs Human Worker的12月现金流NPV
2. **基尼系数**随时间变化
3. **市场价格分布**（均值±标准差）
4. **Agent存活数 + A2A交易网络密度**

**Agent经济价值**：Agent-as-Worker有初始部署成本但推理成本远低于人类工资。

In [ ]:
# TODO 6：Agent经济价值分析（NPV/IRR）+ 可视化
# 提示：
#   1. NPV: Agent现金流转(部署成本+12月净收入) vs Human(12月净收入), 月贴现率=0.10/12
#   2. IRR: npf.irr(agent_cashflows)
#   3. matplotlib 4子图: 基尼/价格/存活/网络密度+A2A

# ===== 你的代码 =====

# raise NotImplementedError

## 8. 天道推演 × 多Agent仿真

本仿真本质是**计算化的天道推演沙盘**：

| 天道推演能力 | 仿真对应 | 涌现产出 |
|-------------|---------|---------|
| 局势感知 | 初始Agent分布与参数 | 初始基尼/价格 |
| 因果链追踪 | Agent行为因果（购买->定价->竞争） | 价格动态 |
| 沙盘模拟（3层） | 20 tick推演 | 时间序列涌现 |
| 概率评估 | 多次运行不同seed | 结果分布 |
| 最优路径推荐 | 对比不同A2A协议参数 | 策略选择 |

**核心洞察**：Agent经济仿真让天道推演从"意识中的沙盘"变为"可计算、可复现的沙盘"。

## 交付物
- [ ] 完成的 starter.ipynb（6个TODO全部填好）
- [ ] 4个子图的仿真结果可视化
- [ ] 一段300字分析：仿真涌现了什么经济现象？推理成本对Agent的影响？网络拓扑说明了什么？